## Human Pose Estimation для детекции курения

Два направления экспериментов:
- **Aspect 1 (Static)**: классификация по скелетным точкам одного кадра
- **Aspect 2 (Temporal)**: анализ динамики движения точек во времени

Структура датасета:
```
smoking_videos/
    smoking/
        smoke/          <- HMDB51: директории с jpg-кадрами
    not_smoking/
        eat/            <- HMDB51: директории с jpg-кадрами
        drink/          <- HMDB51: директории с jpg-кадрами
        meva/           <- наши mp4
    test_smoking/
    test_not_smoking/
```

## 0. Установка зависимостей

In [ ]:
# !pip install ultralytics opencv-python numpy pandas scikit-learn matplotlib seaborn torch torchvision torchaudio tqdm imbalanced-learn scipy

## 1. Импорты

In [ ]:
import os
import cv2
import pickle
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict, deque

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from scipy.signal import medfilt

from tqdm.notebook import tqdm

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score,
    precision_recall_curve, auc
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

## 2. Конфигурация

In [ ]:
DATA_DIR      = Path('smoking_videos')
KEYPOINTS_DIR = Path('keypoints_cache')
MODELS_DIR    = Path('hpe_models')
KEYPOINTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_SEED     = 42
TEST_RATIO      = 0.2

# параметры временного окна для aspect 2
TEMPORAL_WINDOW = 30
TEMPORAL_STRIDE = 15

# COCO-17 keypoints (YOLOv8-pose)
YOLO_POSE_LANDMARKS = 17

CLASS_NAMES = {0: 'not_smoking', 1: 'smoking'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

# COCO-17: 0-nose, 1-l_eye, 2-r_eye, 3-l_ear, 4-r_ear,
#          5-l_shoulder, 6-r_shoulder, 7-l_elbow, 8-r_elbow,
#          9-l_wrist, 10-r_wrist, 11-l_hip, 12-r_hip,
#          13-l_knee, 14-r_knee, 15-l_ankle, 16-r_ankle

# подграф для детекции курения (COCO-индексы)
SMOKING_GRAPH_EDGES = [
    (0, 1), (0, 2),
    (1, 3), (2, 4),
    (3, 5), (4, 6),
    (5, 6),
    (5, 7), (6, 8),
    (7, 9), (8, 10),
    (0, 9), (0, 10),
]
N_JOINTS    = 9
# отображение COCO-индексов -> локальные индексы подграфа
JOINT_REMAP_COCO = {0: 0, 3: 1, 4: 2, 5: 3, 6: 4, 7: 5, 8: 6, 9: 7, 10: 8}

COCO_FULL_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]

## 3. Загрузка датасета

In [ ]:
def collect_samples(data_dir, split='train'):
    samples = []
    if split == 'train':
        class_dirs = {1: data_dir / 'smoking', 0: data_dir / 'not_smoking'}
    else:
        class_dirs = {1: data_dir / 'test_smoking', 0: data_dir / 'test_not_smoking'}

    for label, class_dir in class_dirs.items():
        if not class_dir.exists():
            continue
        for source_dir in class_dir.iterdir():
            if not source_dir.is_dir():
                continue
            for mp4 in source_dir.glob('*.mp4'):
                samples.append((mp4, label, 'mp4'))
            for video_dir in source_dir.iterdir():
                if not video_dir.is_dir():
                    continue
                jpgs = list(video_dir.glob('*.jpg')) + list(video_dir.glob('*.jpeg'))
                if jpgs:
                    samples.append((video_dir, label, 'jpg_dir'))
    return samples


def split_train_test(data_dir, test_ratio=TEST_RATIO, seed=RANDOM_SEED):
    random.seed(seed)
    for class_name in ['smoking', 'not_smoking']:
        src_dir  = data_dir / class_name
        test_dir = data_dir / f'test_{class_name}'
        test_dir.mkdir(exist_ok=True)
        if not src_dir.exists():
            continue
        all_items = []
        for source_dir in src_dir.iterdir():
            if not source_dir.is_dir():
                continue
            for f in source_dir.glob('*.mp4'):
                all_items.append(('mp4', source_dir.name, f))
            for d in source_dir.iterdir():
                if d.is_dir() and list(d.glob('*.jpg')):
                    all_items.append(('jpg_dir', source_dir.name, d))
        already_in_test = sum(
            len(list((test_dir / src).glob('*')))
            for src in ['smoke', 'eat', 'drink', 'meva']
            if (test_dir / src).exists()
        )
        if already_in_test > 0:
            print(f'  {class_name}: test уже заполнен ({already_in_test} элементов), пропускаем')
            continue
        n_test = max(1, int(len(all_items) * test_ratio))
        test_items = random.sample(all_items, n_test)
        for src_type, source_name, path in tqdm(test_items, desc=f'  split {class_name}', leave=False):
            dest_source = test_dir / source_name
            dest_source.mkdir(exist_ok=True)
            shutil.move(str(path), str(dest_source / path.name))
        print(f'  {class_name}: {len(all_items) - n_test} train, {n_test} test')


print('=== разделение на train/test ===')
split_train_test(DATA_DIR)

train_samples = collect_samples(DATA_DIR, split='train')
test_samples  = collect_samples(DATA_DIR, split='test')

print(f'\ntrain: {len(train_samples)} видео'
      f' ({sum(l for _,l,_ in train_samples)} smoking,'
      f' {sum(1-l for _,l,_ in train_samples)} not_smoking)')
print(f'test:  {len(test_samples)} видео'
      f' ({sum(l for _,l,_ in test_samples)} smoking,'
      f' {sum(1-l for _,l,_ in test_samples)} not_smoking)')

df_stat = pd.DataFrame([
    {'split': 'train', 'label': CLASS_NAMES[l], 'source_type': t}
    for _, l, t in train_samples
] + [
    {'split': 'test', 'label': CLASS_NAMES[l], 'source_type': t}
    for _, l, t in test_samples
])
print()
print(df_stat.groupby(['split', 'label', 'source_type']).size().to_string())

## 4. Извлечение скелетных точек YOLOv8-pose

In [ ]:
# COCO-17 индексы, используемые для признаков курения:
# 0  - nose
# 3  - left_ear,    4  - right_ear
# 5  - left_shoulder, 6  - right_shoulder
# 7  - left_elbow,    8  - right_elbow
# 9  - left_wrist,    10 - right_wrist

yolo_pose_model_global = YOLO('yolov8x-pose.pt')


def normalize_kps_to_bbox(kps17, box_xyxy):
    # нормирует координаты кейпоинтов относительно bounding box человека
    # это делает признаки инвариантными к расстоянию от камеры
    # kps17: (17, 3) — x_px, y_px, conf (до нормировки на кадр)
    # box_xyxy: [x1, y1, x2, y2] в пикселях
    x1, y1, x2, y2 = box_xyxy
    bw = max(x2 - x1, 1e-8)
    bh = max(y2 - y1, 1e-8)
    out = kps17.copy()
    out[:, 0] = (kps17[:, 0] - x1) / bw
    out[:, 1] = (kps17[:, 1] - y1) / bh
    return out.astype(np.float32)


def extract_all_persons_kps_frame(frame_bgr, model):
    # возвращает список [(kps17_bbox_norm, box_xyxy), ...] для всех людей в кадре
    # kps17_bbox_norm: (17, 3) — координаты нормированы относительно bbox каждого человека
    h, w = frame_bgr.shape[:2]
    res  = model(frame_bgr, verbose=False)
    persons = []
    if (res and res[0].keypoints is not None
            and len(res[0].keypoints.data) > 0):
        kps_all   = res[0].keypoints.data.cpu().numpy()
        boxes_all = res[0].boxes.xyxy.cpu().numpy()
        for kps17, box in zip(kps_all, boxes_all):
            kps17_px = kps17.copy()
            kps17_norm_bbox = normalize_kps_to_bbox(kps17_px, box)
            persons.append((kps17_norm_bbox, box))
    return persons


def extract_kps_yolopose_frame(frame_bgr, model):
    # возвращает (17, 3): координаты нормированы относительно bbox крупнейшего человека
    # используется для совместимости с пайплайном обучения (один человек в кадре)
    h, w  = frame_bgr.shape[:2]
    persons = extract_all_persons_kps_frame(frame_bgr, model)
    if not persons:
        return np.zeros((YOLO_POSE_LANDMARKS, 3), dtype=np.float32)
    boxes_all = np.array([p[1] for p in persons])
    areas     = [(b[2]-b[0])*(b[3]-b[1]) for b in boxes_all]
    chosen    = int(np.argmax(areas))
    return persons[chosen][0]


def extract_kps_from_mp4_yolo(video_path, model, max_frames=None):
    cap        = cv2.VideoCapture(str(video_path))
    frames_kps = []
    idx        = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or (max_frames and idx >= max_frames):
            break
        frames_kps.append(extract_kps_yolopose_frame(frame, model))
        idx += 1
    cap.release()
    return np.array(frames_kps) if frames_kps else np.zeros((1, YOLO_POSE_LANDMARKS, 3))


def extract_kps_from_jpg_dir_yolo(video_dir, model, max_frames=None):
    jpgs = sorted(video_dir.glob('*.jpg')) + sorted(video_dir.glob('*.jpeg'))
    if max_frames:
        jpgs = jpgs[:max_frames]
    frames_kps = []
    for jpg in jpgs:
        frame = cv2.imread(str(jpg))
        if frame is None:
            continue
        frames_kps.append(extract_kps_yolopose_frame(frame, model))
    return np.array(frames_kps) if frames_kps else np.zeros((1, YOLO_POSE_LANDMARKS, 3))


def extract_kps(path, source_type, model, max_frames=None):
    if source_type == 'mp4':
        return extract_kps_from_mp4_yolo(path, model, max_frames)
    return extract_kps_from_jpg_dir_yolo(path, model, max_frames)


def cache_keypoints(samples, model, extractor_name='yolopose', max_frames=None):
    all_kps    = []
    all_labels = []
    n_cached = 0
    n_new    = 0

    for path, label, source_type in tqdm(samples, desc=f'  кейпоинты ({extractor_name})'):
        cache_key  = f'{Path(path).stem}_{source_type}_{extractor_name}'
        cache_path = KEYPOINTS_DIR / f'{cache_key}.npy'

        if cache_path.exists():
            kps = np.load(str(cache_path))
            n_cached += 1
        else:
            kps = extract_kps(path, source_type, model, max_frames)
            np.save(str(cache_path), kps)
            n_new += 1

        all_kps.append(kps)
        all_labels.append(label)

    print(f'  из кеша: {n_cached}, извлечено заново: {n_new}')
    return all_kps, all_labels


print('train кейпоинты:')
train_kps, train_labels = cache_keypoints(train_samples, yolo_pose_model_global)
print('test кейпоинты:')
test_kps, test_labels = cache_keypoints(test_samples, yolo_pose_model_global)

## 5. Aspect 1 — Статические признаки

In [ ]:
def euclidean_distance(p1, p2):
    return np.sqrt(np.sum((p1 - p2) ** 2))


def angle_three_points(a, b, c):
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))


def extract_static_features(frame_kps):
    # frame_kps: (17, 3) — координаты нормированы относительно bbox человека
    # все расстояния нормированы на ширину плеч — абсолютные координаты не используются
    nose       = frame_kps[0,  :2]
    l_ear      = frame_kps[3,  :2]
    r_ear      = frame_kps[4,  :2]
    l_shoulder = frame_kps[5,  :2]
    r_shoulder = frame_kps[6,  :2]
    l_elbow    = frame_kps[7,  :2]
    r_elbow    = frame_kps[8,  :2]
    l_wrist    = frame_kps[9,  :2]
    r_wrist    = frame_kps[10, :2]

    sw = euclidean_distance(l_shoulder, r_shoulder) + 1e-8

    # расстояние запястье-нос / ширина плеч (относительный признак)
    d_lw_nose = euclidean_distance(l_wrist, nose) / sw
    d_rw_nose = euclidean_distance(r_wrist, nose) / sw

    # высота запястья относительно плеча / ширина плеч (относительный признак)
    lw_rel_height = (l_wrist[1] - l_shoulder[1]) / sw
    rw_rel_height = (r_wrist[1] - r_shoulder[1]) / sw

    # расстояние запястье-ухо / ширина плеч
    d_lw_lear = euclidean_distance(l_wrist, l_ear) / sw
    d_rw_rear = euclidean_distance(r_wrist, r_ear) / sw

    # расстояние запястье-плечо / ширина плеч
    d_lw_lsh  = euclidean_distance(l_wrist, l_shoulder) / sw
    d_rw_rsh  = euclidean_distance(r_wrist, r_shoulder) / sw

    return np.array([
        d_lw_nose,
        d_rw_nose,
        min(d_lw_nose, d_rw_nose),
        angle_three_points(l_elbow, l_wrist, nose),
        angle_three_points(r_elbow, r_wrist, nose),
        lw_rel_height,
        rw_rel_height,
        d_lw_lear,
        d_rw_rear,
        angle_three_points(l_shoulder, l_elbow, l_wrist),
        angle_three_points(r_shoulder, r_elbow, r_wrist),
        d_lw_lsh,
        d_rw_rsh,
        frame_kps[9,  2],
        frame_kps[10, 2],
    ], dtype=np.float32)


def build_static_dataset(all_kps, all_labels):
    X, y = [], []
    for kps_seq, label in zip(all_kps, all_labels):
        for frame_kps in kps_seq:
            X.append(extract_static_features(frame_kps))
            y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)


X_train_static, y_train_static = build_static_dataset(train_kps, train_labels)
X_test_static,  y_test_static  = build_static_dataset(test_kps,  test_labels)
print(f'train static: {X_train_static.shape}, test static: {X_test_static.shape}')
print(f'train class distribution: {np.bincount(y_train_static)}')
print(f'test  class distribution: {np.bincount(y_test_static)}')

### 5.1 Baseline-классификаторы (RF, SVM) — без SMOTE

In [ ]:
def evaluate_static_classifiers(X_train, y_train, X_test, y_test):
    classifiers = {
        'RandomForest': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
        ]),
        'GradientBoosting': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', GradientBoostingClassifier(n_estimators=200, random_state=42))
        ]),
        'SVM-RBF': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(kernel='rbf', probability=True, random_state=42))
        ]),
    }

    results = {}
    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]
        results[name] = {
            'f1':      f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_prob),
            'clf':     clf,
        }
        print(f'\n--- {name} ---')
        print(classification_report(y_test, y_pred,
                                    target_names=['not_smoking', 'smoking'],
                                    zero_division=0))
        print(f'ROC-AUC: {results[name]["roc_auc"]:.4f}')
        with open(MODELS_DIR / f'static_{name}.pkl', 'wb') as fh:
            pickle.dump(clf, fh)
        print(f'  saved: static_{name}.pkl')

    return results


static_results = evaluate_static_classifiers(
    X_train_static, y_train_static,
    X_test_static,  y_test_static
)

### 5.2 GBM + SMOTE

In [ ]:
gbm_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    GradientBoostingClassifier(
                   n_estimators=300, learning_rate=0.05,
                   max_depth=5, subsample=0.8, random_state=42))
])
gbm_smote.fit(X_train_static, y_train_static)
y_pred_gbm = gbm_smote.predict(X_test_static)
y_prob_gbm = gbm_smote.predict_proba(X_test_static)[:, 1]
static_results['GBM+SMOTE'] = {
    'f1':      f1_score(y_test_static, y_pred_gbm, zero_division=0),
    'roc_auc': roc_auc_score(y_test_static, y_prob_gbm),
    'clf':     gbm_smote,
}
print('\n--- GBM+SMOTE ---')
print(classification_report(y_test_static, y_pred_gbm,
                             target_names=['not_smoking', 'smoking'], zero_division=0))
print(f'ROC-AUC: {roc_auc_score(y_test_static, y_prob_gbm):.4f}')
with open(MODELS_DIR / 'static_GBM+SMOTE.pkl', 'wb') as fh:
    pickle.dump(gbm_smote, fh)
print('  saved: static_GBM+SMOTE.pkl')

### 5.3 MLP на статических признаках

In [ ]:
class StaticPoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class SmokingMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train, y_train, X_test, y_test, epochs=50, lr=1e-3, batch_size=256):
    scaler   = StandardScaler()
    X_tr     = scaler.fit_transform(X_train)
    X_te     = scaler.transform(X_test)

    train_loader = DataLoader(StaticPoseDataset(X_tr, y_train),
                              batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(StaticPoseDataset(X_te, y_test),
                              batch_size=batch_size)

    model     = SmokingMLP(input_dim=X_tr.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history   = defaultdict(list)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(yb.numpy())

        history['train_loss'].append(total_loss / len(train_loader))
        history['val_f1'].append(f1_score(trues, preds, zero_division=0))

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | loss: {history["train_loss"][-1]:.4f}'
                  f' | val_f1: {history["val_f1"][-1]:.4f}')

    torch.save({'model_state': model.state_dict(), 'scaler': scaler,
                'input_dim': X_tr.shape[1]},
               MODELS_DIR / 'static_MLP.pt')
    print('  saved: static_MLP.pt')
    return model, scaler, history


mlp_model, mlp_scaler, mlp_history = train_mlp(
    X_train_static, y_train_static,
    X_test_static,  y_test_static
)

## 6. Aspect 2 — Временная динамика

### 6.1 FFT-периодичность + GBM + SMOTE

In [ ]:
def compute_wrist_to_nose_signal_coco(kps_seq):
    # kps_seq: (T, 17, 3) — координаты нормированы относительно bbox каждого человека
    # все расстояния нормированы на ширину плеч
    nose       = kps_seq[:, 0,  :2]
    l_wrist    = kps_seq[:, 9,  :2]
    r_wrist    = kps_seq[:, 10, :2]
    l_shoulder = kps_seq[:, 5,  :2]
    r_shoulder = kps_seq[:, 6,  :2]
    sw = np.linalg.norm(l_shoulder - r_shoulder, axis=1) + 1e-8
    return np.minimum(
        np.linalg.norm(l_wrist - nose, axis=1) / sw,
        np.linalg.norm(r_wrist - nose, axis=1) / sw
    )


# алиас для обратной совместимости
compute_wrist_to_nose_signal = compute_wrist_to_nose_signal_coco


def extract_periodicity_features(signal, fps=30):
    T     = len(signal)
    fft   = np.abs(np.fft.rfft(signal - signal.mean()))
    freqs = np.fft.rfftfreq(T, d=1.0 / fps)
    mask  = (freqs >= 0.1) & (freqs <= 2.0)
    fft_m = fft.copy()
    fft_m[~mask] = 0
    dom_freq  = freqs[np.argmax(fft_m)] if mask.any() else 0.0
    dom_power = fft_m.max()
    total_p   = fft.sum() + 1e-8
    thr       = signal.mean() - 0.5 * signal.std()
    n_appr    = int((np.diff((signal < thr).astype(int)) > 0).sum())
    return np.array([
        signal.mean(), signal.std(), signal.min(),
        signal.max() - signal.min(),
        dom_freq, dom_power, dom_power / total_p,
        float(n_appr),
    ], dtype=np.float32)


def build_periodicity_dataset(all_kps, all_labels,
                               window=90, stride=45, fps=30, min_len=16):
    X, y = [], []
    for kps_seq, label in zip(all_kps, all_labels):
        T = len(kps_seq)
        if T < min_len:
            continue
        for start in range(0, T - window + 1, stride):
            X.append(extract_periodicity_features(
                compute_wrist_to_nose_signal_coco(kps_seq[start:start+window]), fps))
            y.append(label)
        if T >= min_len and T < window:
            X.append(extract_periodicity_features(
                compute_wrist_to_nose_signal_coco(kps_seq), fps))
            y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)


X_train_period, y_train_period = build_periodicity_dataset(train_kps, train_labels)
X_test_period,  y_test_period  = build_periodicity_dataset(test_kps,  test_labels)
print(f'train periodicity: {X_train_period.shape}')
print(f'test  periodicity: {X_test_period.shape}')

period_clf = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    GradientBoostingClassifier(
                   n_estimators=400, learning_rate=0.05,
                   max_depth=4, subsample=0.8, random_state=42))
])
period_clf.fit(X_train_period, y_train_period)
y_pred_period = period_clf.predict(X_test_period)
y_prob_period = period_clf.predict_proba(X_test_period)[:, 1]
print('\n--- GBM+SMOTE (periodicity) ---')
print(classification_report(y_test_period, y_pred_period,
                             target_names=['not_smoking', 'smoking'], zero_division=0))
print(f'ROC-AUC: {roc_auc_score(y_test_period, y_prob_period):.4f}')
with open(MODELS_DIR / 'periodicity_GBM.pkl', 'wb') as fh:
    pickle.dump(period_clf, fh)
print('  saved: periodicity_GBM.pkl')

### 6.2 Temporal нейросети (ST-GCN, Pose-Transformer, TemporalCNN)

In [ ]:
def build_adjacency_matrix(edges, n_joints):
    A = np.eye(n_joints, dtype=np.float32)
    for i, j in edges:
        if i < n_joints and j < n_joints:
            A[i, j] = A[j, i] = 1
    D = np.diag(A.sum(axis=1) ** -0.5)
    return D @ A @ D


def remap_keypoints_coco(kps_seq):
    # kps_seq: (T, 17, 3) -> (T, N_JOINTS, 3)
    # берём только 9 ключевых суставов, используем x_norm, y_norm, conf
    indices = sorted(JOINT_REMAP_COCO.keys())
    return kps_seq[:, indices, :].astype(np.float32)


def build_temporal_sequences(all_kps, all_labels,
                              window=TEMPORAL_WINDOW, stride=TEMPORAL_STRIDE,
                              min_len=16):
    sequences, seq_labels = [], []
    for kps_seq, label in zip(all_kps, all_labels):
        T = len(kps_seq)
        if T < min_len:
            continue
        for start in range(0, T - window + 1, stride):
            sequences.append(remap_keypoints_coco(kps_seq[start:start + window]))
            seq_labels.append(label)
        if T >= min_len and T < window:
            clip = remap_keypoints_coco(kps_seq)
            pad  = np.zeros((window - T, N_JOINTS, 3), dtype=np.float32)
            sequences.append(np.concatenate([clip, pad], axis=0))
            seq_labels.append(label)
    return sequences, np.array(seq_labels)


train_seqs, train_seq_labels = build_temporal_sequences(train_kps, train_labels)
test_seqs,  test_seq_labels  = build_temporal_sequences(test_kps,  test_labels)
print(f'train sequences: {len(train_seqs)}, test sequences: {len(test_seqs)}')
if train_seqs:
    print(f'sequence shape: {train_seqs[0].shape}')

In [ ]:
class TemporalPoseDataset(Dataset):
    def __init__(self, sequences, labels):
        self.X = [torch.tensor(s, dtype=torch.float32) for s in sequences]
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def eval_temporal_model(model, test_seqs, test_labels, batch_size=32, return_probs=False):
    test_loader = DataLoader(TemporalPoseDataset(test_seqs, test_labels),
                             batch_size=batch_size)
    model.eval()
    probs, trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = torch.softmax(model(xb.to(DEVICE)), dim=1)[:, 1]
            probs.extend(p.cpu().numpy())
            trues.extend(yb.numpy())
    probs = np.array(probs)
    if return_probs:
        return probs, np.array(trues)
    preds = probs.round().astype(int)
    return {
        'f1':      f1_score(trues, preds, zero_division=0),
        'roc_auc': roc_auc_score(trues, probs),
    }


def train_temporal_model(model, train_seqs, train_labels, test_seqs, test_labels,
                          model_name='model', epochs=30, lr=1e-3, batch_size=32):
    train_loader = DataLoader(TemporalPoseDataset(train_seqs, train_labels),
                              batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TemporalPoseDataset(test_seqs, test_labels),
                              batch_size=batch_size)

    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader)
    )
    criterion = nn.CrossEntropyLoss()
    history   = defaultdict(list)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        metrics = eval_temporal_model(model, test_seqs, test_labels, batch_size)
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_f1'].append(metrics['f1'])
        history['val_roc_auc'].append(metrics['roc_auc'])

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1}/{epochs} | loss: {history["train_loss"][-1]:.4f}'
                  f' | f1: {metrics["f1"]:.4f} | roc_auc: {metrics["roc_auc"]:.4f}')

    torch.save({'model_state': model.state_dict(), 'model_name': model_name},
               MODELS_DIR / f'temporal_{model_name}.pt')
    print(f'  saved: temporal_{model_name}.pt')
    return model, history

In [ ]:
class STGCNLayer(nn.Module):
    def __init__(self, in_channels, out_channels, A, temporal_kernel=9):
        super().__init__()
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        pad = (temporal_kernel - 1) // 2
        self.gcn = nn.Linear(in_channels, out_channels)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=(temporal_kernel, 1), padding=(pad, 0))
        )
        self.bn  = nn.BatchNorm2d(out_channels)
        self.res = nn.Conv2d(in_channels, out_channels, 1) \
                   if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        B, C, T, V = x.shape
        x_sp = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)
        x_sp = torch.matmul(self.A, x_sp)
        x_sp = self.gcn(x_sp)
        C2   = x_sp.shape[-1]
        x_sp = x_sp.view(B, T, V, C2).permute(0, 3, 1, 2)
        return F.relu(self.bn(self.tcn(x_sp)) + self.res(x))


class STGCN(nn.Module):
    def __init__(self, in_channels=3, num_classes=2, n_joints=N_JOINTS, A=None):
        super().__init__()
        if A is None:
            A = build_adjacency_matrix(SMOKING_GRAPH_EDGES, n_joints)
        self.data_bn = nn.BatchNorm1d(in_channels * n_joints)
        self.layers  = nn.ModuleList([
            STGCNLayer(in_channels, 64,  A),
            STGCNLayer(64, 64,  A),
            STGCNLayer(64, 128, A),
            STGCNLayer(128, 128, A),
        ])
        self.pool    = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        B, T, V, C = x.shape
        x = x.permute(0, 3, 1, 2)
        xb = x.permute(0, 2, 3, 1).contiguous().view(B * T, V * C)
        xb = self.data_bn(xb).view(B, T, V * C)
        x  = xb.view(B, T, V, C).permute(0, 3, 1, 2)
        for layer in self.layers:
            x = layer(x)
        return self.fc(self.dropout(self.pool(x).squeeze(-1).squeeze(-1)))


class PoseTransformer(nn.Module):
    def __init__(self, n_joints=N_JOINTS, in_channels=3, d_model=128,
                 nhead=4, num_layers=4, num_classes=2, max_seq_len=300):
        super().__init__()
        self.input_proj  = nn.Linear(n_joints * in_channels, d_model)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        enc_layer        = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, d_model))
        self.fc          = nn.Linear(d_model, num_classes)
        self.dropout     = nn.Dropout(0.2)

    def forward(self, x):
        B, T, V, C = x.shape
        x   = self.input_proj(x.view(B, T, V * C))
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x   = x + self.pos_emb(pos)
        cls = self.cls_token.expand(B, -1, -1)
        x   = self.transformer(torch.cat([cls, x], dim=1))
        return self.fc(self.dropout(x[:, 0, :]))


class TemporalCNN(nn.Module):
    def __init__(self, n_joints=N_JOINTS, in_channels=3, num_classes=2):
        super().__init__()
        inp = n_joints * in_channels
        self.conv_layers = nn.Sequential(
            nn.Conv1d(inp, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        B, T, V, C = x.shape
        return self.fc(self.conv_layers(x.view(B, T, V * C).permute(0, 2, 1)).squeeze(-1))


A_mat = build_adjacency_matrix(SMOKING_GRAPH_EDGES, N_JOINTS)

print('--- ST-GCN ---')
stgcn, stgcn_hist = train_temporal_model(
    STGCN(in_channels=3, num_classes=2, n_joints=N_JOINTS, A=A_mat),
    train_seqs, train_seq_labels, test_seqs, test_seq_labels,
    model_name='STGCN'
)

print('\n--- Pose-Transformer ---')
pt, pt_hist = train_temporal_model(
    PoseTransformer(n_joints=N_JOINTS, in_channels=3, d_model=128),
    train_seqs, train_seq_labels, test_seqs, test_seq_labels,
    model_name='PoseTransformer'
)

print('\n--- Temporal CNN ---')
tcnn, tcnn_hist = train_temporal_model(
    TemporalCNN(n_joints=N_JOINTS, in_channels=3),
    train_seqs, train_seq_labels, test_seqs, test_seq_labels,
    model_name='TemporalCNN'
)

## 7. Визуализация результатов

In [ ]:
def plot_training_history(histories, names):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    for hist, name in zip(histories, names):
        ax1.plot(hist['train_loss'],   label=name)
        ax2.plot(hist['val_f1'],       label=name)
        ax3.plot(hist['val_roc_auc'],  label=name)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss');  ax1.set_title('Training Loss');  ax1.legend()
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val F1');      ax2.set_title('Validation F1');  ax2.legend()
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Val ROC-AUC'); ax3.set_title('Val ROC-AUC');    ax3.legend()
    plt.tight_layout(); plt.show()


def plot_results_summary(results_dict):
    names = list(results_dict.keys())
    f1s   = [results_dict[n]['f1']      for n in names]
    aucs  = [results_dict[n]['roc_auc'] for n in names]
    x     = np.arange(len(names))
    w     = 0.35
    fig, ax = plt.subplots(figsize=(14, 5))
    b1 = ax.bar(x - w/2, f1s,  w, label='F1 (smoking)')
    b2 = ax.bar(x + w/2, aucs, w, label='ROC-AUC')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title('Model Comparison: Smoking Detection via HPE'); ax.legend()
    for b in list(b1) + list(b2):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'{b.get_height():.3f}', ha='center', fontsize=8)
    plt.tight_layout(); plt.show()


plot_training_history(
    [mlp_history, stgcn_hist, pt_hist, tcnn_hist],
    ['MLP', 'ST-GCN', 'Pose-Transformer', 'TemporalCNN']
)

stgcn_metrics = eval_temporal_model(stgcn, test_seqs, test_seq_labels)
pt_metrics    = eval_temporal_model(pt,    test_seqs, test_seq_labels)
tcnn_metrics  = eval_temporal_model(tcnn,  test_seqs, test_seq_labels)

summary = {
    'RF':                static_results.get('RandomForest',    {'f1': 0, 'roc_auc': 0}),
    'SVM':               static_results.get('SVM-RBF',         {'f1': 0, 'roc_auc': 0}),
    'GBM+SMOTE':         static_results.get('GBM+SMOTE',       {'f1': 0, 'roc_auc': 0}),
    'MLP':               {'f1': f1_score(y_test_static,
                                         mlp_model(torch.tensor(
                                             mlp_scaler.transform(X_test_static),
                                             dtype=torch.float32).to(DEVICE)
                                         ).argmax(1).cpu().numpy(),
                                         zero_division=0),
                          'roc_auc': roc_auc_score(
                              y_test_static,
                              torch.softmax(mlp_model(torch.tensor(
                                  mlp_scaler.transform(X_test_static),
                                  dtype=torch.float32).to(DEVICE)), dim=1
                              )[:, 1].detach().cpu().numpy()
                          )},
    'GBM+SMOTE (period)':{'f1':      f1_score(y_test_period, y_pred_period, zero_division=0),
                          'roc_auc': roc_auc_score(y_test_period, y_prob_period)},
    'ST-GCN':            stgcn_metrics,
    'Pose-Transformer':  pt_metrics,
    'TemporalCNN':       tcnn_metrics,
}
plot_results_summary(summary)

print('\n=== итоговые метрики ===')
for name, m in summary.items():
    print(f'{name:25s}  f1={m["f1"]:.4f}  roc_auc={m["roc_auc"]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

def get_probs_labels(name):
    if name == 'RF':
        clf = static_results['RandomForest']['clf']
        return clf.predict_proba(X_test_static)[:, 1], y_test_static
    if name == 'SVM':
        clf = static_results['SVM-RBF']['clf']
        return clf.predict_proba(X_test_static)[:, 1], y_test_static
    if name == 'GBM+SMOTE':
        return y_prob_gbm, y_test_static
    if name == 'MLP':
        mlp_model.eval()
        with torch.no_grad():
            X_te = torch.tensor(mlp_scaler.transform(X_test_static), dtype=torch.float32).to(DEVICE)
            probs = torch.softmax(mlp_model(X_te), dim=1)[:, 1].cpu().numpy()
        return probs, y_test_static
    if name == 'GBM+SMOTE (period)':
        return y_prob_period, y_test_period
    if name == 'ST-GCN':
        return eval_temporal_model(stgcn, test_seqs, test_seq_labels, return_probs=True)
    if name == 'Pose-Transformer':
        return eval_temporal_model(pt, test_seqs, test_seq_labels, return_probs=True)
    if name == 'TemporalCNN':
        return eval_temporal_model(tcnn, test_seqs, test_seq_labels, return_probs=True)


model_names = ['RF', 'SVM', 'GBM+SMOTE', 'MLP',
               'GBM+SMOTE (period)', 'ST-GCN', 'Pose-Transformer', 'TemporalCNN']

for ax, name in zip(axes, model_names):
    probs, labels = get_probs_labels(name)
    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, lw=2)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{name}\nPR-AUC = {pr_auc:.4f}')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision-Recall curves (class: smoking)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Выводы

| Aspect | Подход | Модели | Особенности |
|--------|--------|--------|-------------|
| 1 (статика) | геометрические признаки кадра | RF, SVM | без SMOTE, быстро |
| 1 (статика) | геометрические признаки кадра | GBM+SMOTE | с балансировкой классов |
| 1 (статика) | геометрические признаки кадра | MLP | нейросеть, без SMOTE |
| 2 (динамика) | FFT-признаки сигнала запястье-нос | GBM+SMOTE | интерпретируемо, периодичность |
| 2 (динамика) | граф + временная свёртка | ST-GCN | структура скелета + время |
| 2 (динамика) | attention по времени | Pose-Transformer | длинные зависимости |
| 2 (динамика) | свёртка по времени | TemporalCNN | быстрый, лёгкий |

## 9. Инструменты трекинга и детекции

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-8)


def compute_distance(box1, box2):
    cx1 = (box1[0] + box1[2]) / 2; cy1 = (box1[1] + box1[3]) / 2
    cx2 = (box2[0] + box2[2]) / 2; cy2 = (box2[1] + box2[3]) / 2
    return np.sqrt((cx1-cx2)**2 + (cy1-cy2)**2)


def box_center(box):
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2], dtype=np.float32)


def split_frame_into_quadrants(frame):
    h, w = frame.shape[:2]
    half_h, half_w = h // 2, w // 2
    quadrants = [
        frame[0:half_h, 0:half_w],
        frame[0:half_h, half_w:w],
        frame[half_h:h, 0:half_w],
        frame[half_h:h, half_w:w],
    ]
    offsets = [(0, 0), (half_w, 0), (0, half_h), (half_w, half_h)]
    return quadrants, offsets, (half_w, half_h)


def scale_detections_to_original(detections, offset, scale_factor):
    if detections is None or len(detections) == 0:
        return []
    scaled = []
    for det in detections:
        boxes = det.boxes
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                scaled.append({
                    'box': [
                        x1 / scale_factor + offset[0],
                        y1 / scale_factor + offset[1],
                        x2 / scale_factor + offset[0],
                        y2 / scale_factor + offset[1],
                    ],
                    'conf': float(box.conf[0].cpu().numpy()),
                    'cls':  int(box.cls[0].cpu().numpy()),
                })
    return scaled


class DetectionTracker:
    def __init__(self, max_age=15, min_hits=3,
                 iou_threshold=0.3, distance_threshold=50):
        self.max_age            = max_age
        self.min_hits           = min_hits
        self.iou_threshold      = iou_threshold
        self.distance_threshold = distance_threshold
        self.tracks             = []
        self.next_id            = 0

    def update(self, detections):
        if len(self.tracks) == 0:
            for det in detections:
                self.tracks.append({
                    'id': self.next_id, 'box': det['box'],
                    'conf': det['conf'], 'cls': det['cls'],
                    'age': 0, 'hits': 1,
                    'conf_history': [det['conf']],
                    'has_smoke': det.get('has_smoke', False),
                })
                self.next_id += 1
            return self.tracks

        if len(detections) == 0:
            for t in self.tracks:
                t['age'] += 1
            self.tracks = [t for t in self.tracks if t['age'] < self.max_age]
            return self.tracks

        iou_matrix = np.zeros((len(self.tracks), len(detections)))
        for i, track in enumerate(self.tracks):
            for j, det in enumerate(detections):
                dist = compute_distance(track['box'], det['box'])
                if dist < self.distance_threshold:
                    iou_matrix[i, j] = compute_iou(track['box'], det['box'])

        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        matched_tracks, matched_dets = set(), set()

        for i, j in zip(row_ind, col_ind):
            if iou_matrix[i, j] > self.iou_threshold:
                self.tracks[i].update({
                    'box': detections[j]['box'],
                    'conf': detections[j]['conf'],
                    'cls': detections[j]['cls'],
                    'age': 0,
                    'has_smoke': detections[j].get('has_smoke', False),
                })
                self.tracks[i]['hits'] += 1
                self.tracks[i]['conf_history'].append(detections[j]['conf'])
                if len(self.tracks[i]['conf_history']) > 10:
                    self.tracks[i]['conf_history'].pop(0)
                matched_tracks.add(i)
                matched_dets.add(j)

        for i, t in enumerate(self.tracks):
            if i not in matched_tracks:
                t['age'] += 1

        for j, det in enumerate(detections):
            if j not in matched_dets:
                self.tracks.append({
                    'id': self.next_id, 'box': det['box'],
                    'conf': det['conf'], 'cls': det['cls'],
                    'age': 0, 'hits': 1,
                    'conf_history': [det['conf']],
                    'has_smoke': det.get('has_smoke', False),
                })
                self.next_id += 1

        self.tracks = [t for t in self.tracks if t['age'] < self.max_age]
        return [t for t in self.tracks if t['hits'] >= self.min_hits]

In [ ]:
class PersonTrack:
    """Трек одного человека с историей кейпоинтов и предсказанием скорости."""

    def __init__(self, track_id, kps17, box, max_history=150):
        self.track_id    = track_id
        self.box         = np.array(box, dtype=np.float32)
        self.age         = 0
        self.hits        = 1
        self.kps_history = deque([kps17], maxlen=max_history)
        self.gbm_prob    = 0.0
        self.stgcn_prob  = 0.0
        # история центров для предсказания velocity
        self.center_history = deque([box_center(box)], maxlen=30)
        # предсказанный центр на следующий кадр
        self.predicted_center = box_center(box).copy()

    def _update_velocity(self):
        # сглаженная скорость по нескольким последним центрам
        if len(self.center_history) >= 3:
            centers = np.array(self.center_history)
            # взвешенное среднее разностей (последние важнее)
            diffs   = np.diff(centers, axis=0)
            weights = np.arange(1, len(diffs) + 1, dtype=np.float32)
            weights /= weights.sum()
            velocity = (diffs * weights[:, None]).sum(axis=0)
        elif len(self.center_history) == 2:
            centers  = np.array(self.center_history)
            velocity = centers[-1] - centers[-2]
        else:
            velocity = np.zeros(2, dtype=np.float32)
        self.predicted_center = np.array(self.center_history[-1]) + velocity

    def update(self, kps17, box):
        self.kps_history.append(kps17)
        self.box  = np.array(box, dtype=np.float32)
        c         = box_center(box)
        self.center_history.append(c)
        self._update_velocity()
        self.age  = 0
        self.hits += 1

    def get_kps_seq(self):
        return np.array(self.kps_history)

    def compute_hpe_scores(self, period_clf, stgcn_model, orig_fps):
        kps_seq = self.get_kps_seq()
        T       = len(kps_seq)
        if T < 16:
            return

        signal        = compute_wrist_to_nose_signal_coco(kps_seq)
        feats         = extract_periodicity_features(signal, fps=orig_fps).reshape(1, -1)
        self.gbm_prob = period_clf.predict_proba(feats)[0, 1]

        seq_r = remap_keypoints_coco(kps_seq[:min(T, TEMPORAL_WINDOW)])
        if len(seq_r) < TEMPORAL_WINDOW:
            pad   = np.zeros((TEMPORAL_WINDOW - len(seq_r), N_JOINTS, 3), dtype=np.float32)
            seq_r = np.concatenate([seq_r, pad], axis=0)
        inp = torch.tensor(seq_r[None], dtype=torch.float32).to(DEVICE)
        stgcn_model.eval()
        with torch.no_grad():
            self.stgcn_prob = torch.softmax(stgcn_model(inp), dim=1)[0, 1].item()


class MultiPersonTracker:
    """Трекер нескольких людей с учётом истории перемещений.

    Для матчинга использует комбинацию IoU и расстояния
    от предсказанной позиции (velocity-предсказание),
    что позволяет корректно переназначать ID при
    пересечении траекторий.
    """

    def __init__(self, max_age=20, min_hits=3,
                 iou_threshold=0.25, distance_threshold=120,
                 kps_history_len=150,
                 velocity_weight=0.4):
        self.max_age          = max_age
        self.min_hits         = min_hits
        self.iou_threshold    = iou_threshold
        self.distance_thr     = distance_threshold
        self.kps_history_len  = kps_history_len
        self.velocity_weight  = velocity_weight  # вклад velocity в стоимость матчинга
        self.tracks           = {}   # track_id -> PersonTrack
        self.next_id          = 0

    def _build_cost_matrix(self, track_list, detections):
        # cost = -(iou) + velocity_weight * norm_velocity_dist
        # чем ниже cost — тем лучше матч
        n_tracks = len(track_list)
        n_dets   = len(detections)
        cost_mat = np.full((n_tracks, n_dets), 1e6, dtype=np.float32)

        for i, track in enumerate(track_list):
            for j, (_, box) in enumerate(detections):
                det_center = box_center(box)
                # расстояние от предсказанного центра трека до центра детекции
                pred_dist  = float(np.linalg.norm(track.predicted_center - det_center))
                if pred_dist >= self.distance_thr:
                    continue
                iou_val    = compute_iou(track.box.tolist(), box)
                # нормируем pred_dist в [0, 1] по порогу
                norm_dist  = pred_dist / (self.distance_thr + 1e-8)
                cost_mat[i, j] = -iou_val + self.velocity_weight * norm_dist

        return cost_mat

    def update(self, detections):
        # detections: list of (kps17_bbox_norm, box_px)
        if not detections:
            for t in self.tracks.values():
                t.age += 1
                t._update_velocity()
            self.tracks = {k: v for k, v in self.tracks.items()
                           if v.age < self.max_age}
            return self.tracks

        track_ids  = list(self.tracks.keys())
        track_list = [self.tracks[tid] for tid in track_ids]

        if not track_list:
            for kps17, box in detections:
                tid = self.next_id
                self.tracks[tid] = PersonTrack(tid, kps17, box,
                                               self.kps_history_len)
                self.next_id += 1
            return self.tracks

        cost_mat = self._build_cost_matrix(track_list, detections)
        row_ind, col_ind = linear_sum_assignment(cost_mat)
        matched_tracks, matched_dets = set(), set()

        for i, j in zip(row_ind, col_ind):
            if cost_mat[i, j] >= 1e5:
                continue
            # принимаем матч, только если IoU выше порога
            iou_val = compute_iou(track_list[i].box.tolist(), detections[j][1])
            if iou_val < self.iou_threshold:
                continue
            kps17, box = detections[j]
            track_list[i].update(kps17, box)
            matched_tracks.add(i)
            matched_dets.add(j)

        for i, track in enumerate(track_list):
            if i not in matched_tracks:
                track.age += 1
                track._update_velocity()

        for j, (kps17, box) in enumerate(detections):
            if j not in matched_dets:
                tid = self.next_id
                self.tracks[tid] = PersonTrack(tid, kps17, box,
                                               self.kps_history_len)
                self.next_id += 1

        self.tracks = {k: v for k, v in self.tracks.items()
                       if v.age < self.max_age}
        return self.tracks

## 10. Загрузка YOLO-моделей детекции

In [ ]:
cigarette_model = YOLO('runs/detect/runs/detect/smoking_yolo26l/weights/best.pt')
smoke_model     = YOLO('runs/detect/smoke_boxes_micro_s/smoking_yolo26s_micro/weights/best.pt')
yolo_pose_model = YOLO('yolov8x-pose.pt')

## 11. Вспомогательные функции инференса

In [ ]:
def slice_video_to_segments(video_path, segment_sec=10, overlap_sec=2):
    """Нарезает видео на сегменты длиной segment_sec с перекрытием overlap_sec.

    Возвращает список (start_frame, end_frame, fps).
    Если длина видео <= segment_sec, возвращает один сегмент.
    """
    cap      = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    seg_frames     = int(segment_sec * fps)
    overlap_frames = int(overlap_sec * fps)
    stride_frames  = max(1, seg_frames - overlap_frames)

    if n_frames <= seg_frames:
        return [(0, n_frames, fps)]

    segments = []
    start    = 0
    while start < n_frames:
        end = min(start + seg_frames, n_frames)
        segments.append((start, end, fps))
        if end == n_frames:
            break
        start += stride_frames
    return segments


def read_segment_frames(video_path, start_frame, end_frame):
    cap    = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    for _ in range(end_frame - start_frame):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


SMOKING_JOINT_INDICES = [0, 3, 4, 5, 6, 7, 8, 9, 10]
SMOKING_JOINT_NAMES   = {
    0: 'nose', 3: 'l_ear', 4: 'r_ear',
    5: 'l_shoulder', 6: 'r_shoulder',
    7: 'l_elbow', 8: 'r_elbow',
    9: 'l_wrist', 10: 'r_wrist',
}

STGCN_EDGE_PAIRS = [
    (0, 3), (0, 4),
    (3, 5), (4, 6),
    (5, 7), (6, 8),
    (7, 9), (8, 10),
    (5, 6),
    (0, 9), (0, 10),
]


def draw_skeleton_on_frame(frame_bgr, kps17_norm, pred_label=None, pred_prob=None):
    # kps17_norm: (17, 3) — координаты нормированы относительно bbox человека
    h, w = frame_bgr.shape[:2]
    vis  = frame_bgr.copy()

    for idx in range(17):
        x = int(kps17_norm[idx, 0] * w)
        y = int(kps17_norm[idx, 1] * h)
        if kps17_norm[idx, 2] > 0.3:
            cv2.circle(vis, (x, y), 3, (180, 180, 180), -1)

    for idx in SMOKING_JOINT_INDICES:
        x = int(kps17_norm[idx, 0] * w)
        y = int(kps17_norm[idx, 1] * h)
        if kps17_norm[idx, 2] > 0.3:
            color = (0, 255, 128) if idx in [9, 10] else (255, 128, 0)
            cv2.circle(vis, (x, y), 6, color, -1)
            cv2.putText(vis, SMOKING_JOINT_NAMES[idx], (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    for i, j in STGCN_EDGE_PAIRS:
        xi, yi, ci = kps17_norm[i]
        xj, yj, cj = kps17_norm[j]
        xi, yi = int(xi * w), int(yi * h)
        xj, yj = int(xj * w), int(yj * h)
        if ci > 0.3 and cj > 0.3:
            cv2.line(vis, (xi, yi), (xj, yj), (100, 200, 255), 2)

    if pred_label is not None:
        label_text = f'{"SMOKING" if pred_label == 1 else "NOT SMOKING"}'
        prob_text  = f'p={pred_prob:.2f}' if pred_prob is not None else ''
        color      = (0, 0, 255) if pred_label == 1 else (0, 200, 0)
        cv2.putText(vis, label_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        cv2.putText(vis, prob_text, (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    return vis

## 12. Инференс на видео — fusion + сигналы по людям

In [ ]:
def run_inference_on_video(
        video_path,
        period_clf, stgcn_model,
        cigarette_model, smoke_model, yolo_pose_model,
        output_dir=Path('fusion_output'),
        fps_override=None,
        yolo_conf=0.2,
        scale_factor=2.0,
        hpe_update_interval=15,
        w_gbm=0.20, w_stgcn=0.20, w_cig=0.40, w_smoke=0.20,
        segment_sec=10, overlap_sec=2,
        max_persons_show=6):
    """Полный инференс: fusion-видео + графики сигналов по каждому человеку.

    Каждый человек в кадре рассматривается независимо.
    Кейпоинты нормированы относительно bbox каждого человека —
    устойчивость к расстоянию от камеры обеспечена.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    video_path = Path(video_path)

    cap      = cv2.VideoCapture(str(video_path))
    orig_fps = fps_override or int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    out_path = output_dir / f'{video_path.stem}_fusion.mp4'
    writer   = cv2.VideoWriter(str(out_path),
                                cv2.VideoWriter_fourcc(*'mp4v'),
                                orig_fps, (width, height))

    segments = slice_video_to_segments(video_path, segment_sec, overlap_sec)
    print(f'видео нарезано на {len(segments)} сегментов')

    # history_len: храним кейпоинты до 30 секунд для HPE-анализа
    person_tracker = MultiPersonTracker(kps_history_len=orig_fps * 30)
    cig_tracker    = DetectionTracker()

    KEY_INDICES      = [0, 3, 4, 5, 6, 7, 8, 9, 10]
    global_frame_idx = 0

    for seg_idx, (start_frame, end_frame, _) in enumerate(segments):
        frames = read_segment_frames(video_path, start_frame, end_frame)
        print(f'  сегмент {seg_idx+1}/{len(segments)}: кадры {start_frame}-{end_frame}')

        for frame in frames:
            h, w = frame.shape[:2]

            # ---- YOLOv8-pose: все люди в кадре, кейпоинты нормированы по bbox ----
            pose_res    = yolo_pose_model(frame, verbose=False)
            person_dets = []

            if (pose_res and pose_res[0].keypoints is not None
                    and len(pose_res[0].keypoints.data) > 0):
                kps_all   = pose_res[0].keypoints.data.cpu().numpy()
                boxes_all = pose_res[0].boxes.xyxy.cpu().numpy()
                for kps17_raw, box in zip(kps_all, boxes_all):
                    # нормировка относительно bbox — инвариантность к расстоянию от камеры
                    kps17_norm = normalize_kps_to_bbox(kps17_raw, box)
                    person_dets.append((kps17_norm, box))

            active_tracks = person_tracker.update(person_dets)

            # ---- обновляем HPE-скоры каждые hpe_update_interval кадров ----
            if global_frame_idx % hpe_update_interval == 0:
                for track in active_tracks.values():
                    if track.hits >= 3:
                        track.compute_hpe_scores(period_clf, stgcn_model, orig_fps)

            # ---- YOLO детекция сигарет/дыма через квадранты ----
            quadrants, offsets, _ = split_frame_into_quadrants(frame)
            all_cig_dets   = []
            all_smoke_dets = []

            for quadrant, offset in zip(quadrants, offsets):
                upscaled  = cv2.resize(quadrant, None,
                                       fx=scale_factor, fy=scale_factor,
                                       interpolation=cv2.INTER_CUBIC)
                cig_res   = cigarette_model(upscaled, conf=yolo_conf, verbose=False)
                smoke_res = smoke_model(upscaled, conf=yolo_conf, verbose=False)
                all_cig_dets.extend(
                    scale_detections_to_original(cig_res, offset, scale_factor))
                all_smoke_dets.extend(
                    scale_detections_to_original(smoke_res, offset, scale_factor))

            merged_cig = []
            for cdet in all_cig_dets:
                best_smoke_iou = 0
                has_smoke      = False
                for sdet in all_smoke_dets:
                    iou = compute_iou(cdet['box'], sdet['box'])
                    if iou > 0.1:
                        has_smoke      = True
                        best_smoke_iou = max(best_smoke_iou, iou)
                boost = 1.0 + 0.3 * best_smoke_iou if has_smoke else 1.0
                merged_cig.append({
                    'box':       cdet['box'],
                    'conf':      min(cdet['conf'] * boost, 1.0),
                    'cls':       cdet['cls'],
                    'has_smoke': has_smoke,
                })

            tracked_cig = cig_tracker.update(merged_cig)

            # ---- fusion для каждого человека в отдельности ----
            person_cig_conf   = {tid: 0.0 for tid in active_tracks}
            person_smoke_conf = {tid: 0.0 for tid in active_tracks}

            for cig_track in tracked_cig:
                for tid, person in active_tracks.items():
                    iou = compute_iou(person.box.tolist(), cig_track['box'])
                    if iou > 0.05:
                        avg_conf   = np.mean(cig_track['conf_history'])
                        final_conf = min(avg_conf + min(cig_track['hits']/10.0, 0.2), 1.0)
                        person_cig_conf[tid]   = max(person_cig_conf[tid], final_conf)
                        person_smoke_conf[tid] = max(person_smoke_conf[tid],
                                                     float(cig_track.get('has_smoke', False)))

            for sdet in all_smoke_dets:
                for tid, person in active_tracks.items():
                    iou = compute_iou(person.box.tolist(), sdet['box'])
                    if iou > 0.05:
                        person_smoke_conf[tid] = max(person_smoke_conf[tid],
                                                     sdet['conf'])

            # ---- рисуем ----
            vis = frame.copy()

            for tid, track in active_tracks.items():
                if track.hits < 3:
                    continue

                # денормируем кейпоинты обратно в пиксели кадра для отрисовки
                x1b, y1b, x2b, y2b = track.box
                bw = x2b - x1b
                bh = y2b - y1b
                kps17_bbox = track.get_kps_seq()[-1].copy()
                kps17_px   = kps17_bbox.copy()
                kps17_px[:, 0] = kps17_bbox[:, 0] * bw + x1b
                kps17_px[:, 1] = kps17_bbox[:, 1] * bh + y1b

                cig_conf   = person_cig_conf[tid]
                smoke_conf = person_smoke_conf[tid]

                fusion_prob  = (w_gbm   * track.gbm_prob   +
                                w_stgcn * track.stgcn_prob  +
                                w_cig   * cig_conf          +
                                w_smoke * smoke_conf)
                fusion_label = int(fusion_prob >= 0.5)

                skel_color = (0, 0, 255) if fusion_label else (100, 200, 255)
                for i, j in COCO_FULL_EDGES:
                    xi, yi, ci = kps17_px[i]
                    xj, yj, cj = kps17_px[j]
                    if ci > 0.3 and cj > 0.3:
                        cv2.line(vis, (int(xi), int(yi)), (int(xj), int(yj)),
                                 skel_color, 2)

                for k in KEY_INDICES:
                    x_k, y_k, c_k = kps17_px[k]
                    if c_k > 0.3:
                        color = (0, 255, 128) if k in [9, 10] else (0, 128, 255)
                        cv2.circle(vis, (int(x_k), int(y_k)), 6, color, -1)

                x1, y1, x2, y2 = map(int, track.box)
                box_color = (0, 0, 255) if fusion_label else (0, 200, 0)
                cv2.rectangle(vis, (x1, y1), (x2, y2), box_color, 2)
                label_lines = [
                    f'ID:{tid} {"SMOKING" if fusion_label else "not smoking"}',
                    f'fus={fusion_prob:.2f} cig={cig_conf:.2f}',
                    f'gbm={track.gbm_prob:.2f} stgcn={track.stgcn_prob:.2f}',
                ]
                for li, line in enumerate(label_lines):
                    cv2.putText(vis, line, (x1, y1 - 8 - li * 16),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.42, box_color, 1)

            for cig_track in tracked_cig:
                x1, y1, x2, y2 = map(int, cig_track['box'])
                color = (0, 255, 0) if cig_track.get('has_smoke') else (0, 255, 255)
                cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
                lbl = f'cig {np.mean(cig_track["conf_history"]):.2f}'
                if cig_track.get('has_smoke'):
                    lbl += ' +smoke'
                cv2.putText(vis, lbl, (x1, y2 + 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

            seg_label = (f'seg {seg_idx+1}/{len(segments)}  '
                         f'frame {global_frame_idx}  '
                         f'persons:{len(active_tracks)}')
            cv2.putText(vis, seg_label, (width - 340, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1)

            writer.write(vis)
            global_frame_idx += 1

            if global_frame_idx % 30 == 0:
                print(f'    frame {global_frame_idx}  persons tracked: {len(active_tracks)}')

    writer.release()
    print(f'\nсохранено: {out_path}')

    # ---- построение графиков сигналов по каждому человеку ----
    valid_tracks = {
        tid: track for tid, track in person_tracker.tracks.items()
        if len(track.kps_history) >= 16
    }
    print(f'треков с достаточной историей: {len(valid_tracks)}')

    if valid_tracks:
        sorted_tracks = sorted(valid_tracks.items(),
                               key=lambda x: len(x[1].kps_history),
                               reverse=True)[:max_persons_show]

        n_persons = len(sorted_tracks)
        fig, axes = plt.subplots(n_persons, 3,
                                  figsize=(18, 4 * n_persons),
                                  squeeze=False)

        for row, (tid, track) in enumerate(sorted_tracks):
            kps_seq = track.get_kps_seq()
            Tp      = len(kps_seq)
            t_axis  = np.arange(Tp) / orig_fps

            signal = compute_wrist_to_nose_signal_coco(kps_seq)

            ax = axes[row, 0]
            ax.plot(t_axis, signal, lw=1.2, color='steelblue')
            ax.set_title(f'ID:{tid}  wrist-nose signal  (кадров: {Tp})',
                         fontsize=10)
            ax.set_ylabel('dist (norm)')
            ax.set_xlabel('time, s')
            ax.grid(True, alpha=0.3)

            thr = signal.mean() - 0.5 * signal.std()
            ax.axhline(thr, ls='--', color='red', lw=0.8, label=f'threshold={thr:.2f}')
            ax.fill_between(t_axis, signal, thr,
                             where=(signal < thr),
                             alpha=0.3, color='red', label='approach')
            ax.legend(fontsize=7)

            fft   = np.abs(np.fft.rfft(signal - signal.mean()))
            freqs = np.fft.rfftfreq(Tp, d=1.0 / orig_fps)

            ax = axes[row, 1]
            ax.plot(freqs, fft, lw=1.2, color='darkorange')
            ax.axvspan(0.1, 2.0, alpha=0.15, color='green', label='0.1-2 Hz (smoking)')
            ax.set_xlim(0, 4)
            ax.set_title(f'ID:{tid}  FFT', fontsize=10)
            ax.set_xlabel('frequency, Hz')
            ax.set_ylabel('amplitude')
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

            mask     = (freqs >= 0.1) & (freqs <= 2.0)
            fft_m    = fft.copy(); fft_m[~mask] = 0
            dom_freq = freqs[np.argmax(fft_m)] if mask.any() else 0.0
            ax.axvline(dom_freq, color='red', lw=1.0, ls='--',
                       label=f'dom={dom_freq:.2f} Hz')
            ax.legend(fontsize=7)

            track.compute_hpe_scores(period_clf, stgcn_model, orig_fps)

            ax = axes[row, 2]
            ax.axis('off')

            feats        = extract_periodicity_features(signal, fps=orig_fps)
            n_approaches = int(feats[7])

            info = [
                f'ID: {tid}',
                f'кадров в треке: {Tp}',
                f'время в кадре: {Tp/orig_fps:.1f} s',
                '',
                f'GBM(period) prob:  {track.gbm_prob:.4f}',
                f'ST-GCN prob:       {track.stgcn_prob:.4f}',
                '',
                f'signal mean:       {signal.mean():.3f}',
                f'signal min:        {signal.min():.3f}',
                f'signal std:        {signal.std():.3f}',
                f'dom frequency:     {dom_freq:.3f} Hz',
                f'approaches:        {n_approaches}',
            ]

            verdict = ('SMOKING' if (track.gbm_prob > 0.5 or track.stgcn_prob > 0.5)
                       else 'not smoking')
            color   = 'red' if verdict == 'SMOKING' else 'green'

            ax.text(0.05, 0.95, verdict, transform=ax.transAxes,
                    fontsize=16, fontweight='bold', color=color,
                    va='top')
            ax.text(0.05, 0.78, '\n'.join(info), transform=ax.transAxes,
                    fontsize=9, va='top', family='monospace')

        plt.suptitle(f'{video_path.name}  —  сигналы по людям',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()

        plot_path = output_dir / f'{video_path.stem}_person_signals.png'
        plt.savefig(str(plot_path), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'сохранено: {plot_path}')

    return str(out_path)


result = run_inference_on_video(
    'smoking_videofor/smoking .drinking/16.mp4',
    period_clf, stgcn,
    cigarette_model, smoke_model, yolo_pose_model,
)